# Análise de Medicamentos — Qual reduz a pressão arterial sem efeitos colaterais?

**Objetivo:** identificar, entre os 5 medicamentos testados (Cardioxina, Energozin, Glucorex, Relaxol, Thermocor), qual tratamento **reduz a pressão arterial** e ao mesmo tempo **anula qualquer efeito colateral** nas demais variáveis (temperatura, glicose, frequência cardíaca e nível de energia).

**Dados:** 5 arquivos CSV, 30 pacientes cada, com medições **Inicial** e **Final** de 5 variáveis.

**Método:**
1. Carregar os dados e calcular o efeito (Final − Inicial) de cada medicamento em cada variável.
2. Testar significância estatística com **teste t pareado** (cada paciente é seu próprio controle).
3. Identificar efeitos colaterais = variáveis, fora a pressão, que mudaram significativamente.
4. Buscar **combinações** de medicamentos cujos efeitos colaterais se cancelam, mantendo a queda de pressão.


## 1. Setup e upload dos arquivos

Execute a célula abaixo e faça o upload dos 5 arquivos `*_dataset.csv`.

In [ ]:
import io, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

MEDICAMENTOS = ['cardioxina', 'energozin', 'glucorex', 'relaxol', 'thermocor']


In [ ]:
# --- Upload no Google Colab ---
try:
    from google.colab import files
    uploaded = files.upload()          # selecione os 5 arquivos *_dataset.csv
    FONTES = {nome: io.BytesIO(conteudo) for nome, conteudo in uploaded.items()}
except ImportError:
    # Fora do Colab: lê os CSVs da pasta local ./data
    import glob, os
    FONTES = {os.path.basename(p): p for p in glob.glob('data/*_dataset.csv')}

print('Arquivos disponiveis:', list(FONTES))


## 2. Carregamento e padronização dos dados

In [ ]:
VARIAVEIS = {
    'Pressao':    'Pressão',
    'Temperatura':'Temperatura',
    'Glicose':    'Glicose',
    'Frequencia': 'Frequência',
    'Energia':    'Nível de Energia',
}

def carregar(fonte):
    df = pd.read_csv(fonte)
    # a coluna "Paciente" aparece duplicada (inicio e fim) -> mantemos apenas a primeira
    df = df.loc[:, ~df.columns.str.startswith('Paciente.')]
    dados = {}
    for curto, prefixo in VARIAVEIS.items():
        col_ini = [c for c in df.columns if c.startswith(prefixo) and 'Inicial' in c][0]
        col_fim = [c for c in df.columns if c.startswith(prefixo) and 'Final'   in c][0]
        dados[f'{curto}_ini'] = df[col_ini].astype(float)
        dados[f'{curto}_fim'] = df[col_fim].astype(float)
        dados[f'{curto}_delta'] = dados[f'{curto}_fim'] - dados[f'{curto}_ini']
    out = pd.DataFrame(dados)
    out.insert(0, 'Paciente', df.iloc[:, 0].values)
    return out

DADOS = {}
for med in MEDICAMENTOS:
    arquivo = next(n for n in FONTES if n.startswith(med))
    DADOS[med] = carregar(FONTES[arquivo])
    print(f'{med:12s} -> {len(DADOS[med])} pacientes')

DADOS['cardioxina'].head()


## 3. Verificação de qualidade dos dados

Antes de concluir qualquer coisa, checamos valores faltantes, duplicados e valores fisiologicamente impossíveis.

In [ ]:
FAIXAS_PLAUSIVEIS = {
    'Pressao':    (70, 220),   # mmHg
    'Temperatura':(34, 42),    # °C
    'Glicose':    (50, 300),   # mg/dL
    'Frequencia': (40, 160),   # bpm
    'Energia':    (0, 10),     # escala
}

linhas = []
for med, df in DADOS.items():
    for var, (lo, hi) in FAIXAS_PLAUSIVEIS.items():
        cols = [f'{var}_ini', f'{var}_fim']
        fora = ((df[cols] < lo) | (df[cols] > hi)).sum().sum()
        linhas.append({'Medicamento': med, 'Variavel': var,
                       'Faltantes': df[cols].isna().sum().sum(),
                       'Fora da faixa': fora})

qualidade = pd.DataFrame(linhas)
print('Total de valores faltantes:', qualidade['Faltantes'].sum())
display(qualidade[qualidade['Fora da faixa'] > 0])


> **Alerta de qualidade (importante para a conclusão):** o `glucorex_dataset.csv` tem 3 pacientes com glicose final travada em **exatamente 40,0 mg/dL**. Isso não é ruído: é um **piso** nos dados, e 40 mg/dL é hipoglicemia grave. A Seção 6 investiga esse padrão, porque ele muda completamente a leitura do Glucorex.

## 4. Efeito de cada medicamento

Cada paciente serve como seu próprio controle, então comparamos Final vs. Inicial com **teste t pareado** (α = 0,05). Como algumas variáveis são ordinais (Nível de Energia, escala 0–10) e outras têm distribuição suspeita, reportamos **também a mediana e o teste de Wilcoxon** — se os dois testes discordarem, é sinal de que outliers estão comandando a média.

In [ ]:
ALFA = 0.05

def efeito(df, var):
    d = df[f'{var}_delta'].dropna()
    t, p_t = stats.ttest_rel(df[f'{var}_fim'], df[f'{var}_ini'])
    ic = stats.t.interval(0.95, len(d)-1, loc=d.mean(), scale=stats.sem(d))
    try:
        p_w = stats.wilcoxon(d).pvalue          # nao-parametrico, robusto a outliers
    except ValueError:
        p_w = np.nan                            # todas as diferencas iguais a zero
    return {'delta_medio': d.mean(), 'delta_mediano': d.median(),
            'IC95_inf': ic[0], 'IC95_sup': ic[1],
            'p_ttest': p_t, 'p_wilcoxon': p_w,
            'cohen_dz': d.mean() / d.std(ddof=1),
            'significativo': p_t < ALFA,
            'discordancia': (p_t < ALFA) != (p_w < ALFA)}

linhas = []
for med, df in DADOS.items():
    for var in VARIAVEIS:
        linhas.append({'Medicamento': med, 'Variavel': var, **efeito(df, var)})

EFEITOS = pd.DataFrame(linhas)

alertas = EFEITOS[EFEITOS['discordancia']]
if len(alertas):
    print('!! t-teste e Wilcoxon discordam (media distorcida por outliers):')
    display(alertas[['Medicamento','Variavel','delta_medio','delta_mediano','p_ttest','p_wilcoxon']])

EFEITOS.drop(columns=['discordancia'])


In [ ]:
# Visão resumida: efeito medio por medicamento x variavel (* = estatisticamente significativo)
tabela = EFEITOS.pivot(index='Medicamento', columns='Variavel', values='delta_medio')[list(VARIAVEIS)]
marca  = EFEITOS.pivot(index='Medicamento', columns='Variavel', values='significativo')[list(VARIAVEIS)]

resumo = tabela.round(2).astype(str) + np.where(marca, ' *', '')
print('Efeito medio (Final - Inicial).  * = p < 0.05\n')
resumo


## 6. Diagnóstico: os dados do Glucorex não são o que a média sugere

A média do efeito do Glucorex sobre a glicose é **−4,95 mg/dL**, o que o faria parecer o antídoto perfeito para qualquer remédio que suba a glicose. Mas o t-teste (p = 0,063) e o Wilcoxon (p = 0,014) **discordam** — sinal clássico de outliers. Vamos olhar a distribuição real.

In [ ]:
g = DADOS['glucorex']['Glicose_delta'].round(2)
print('Distribuicao dos deltas de glicose do Glucorex:')
display(g.value_counts().sort_index().rename('n pacientes').to_frame())
print(f'media = {g.mean():.2f}   mediana = {g.median():.2f}')

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
a1.hist(g, bins=25, color='#e76f51'); a1.axvline(g.mean(), color='black', ls='--', label=f'media {g.mean():.2f}')
a1.axvline(g.median(), color='#2a9d8f', lw=2, label=f'mediana {g.median():.2f}')
a1.set_title('Glucorex: delta de glicose'); a1.set_xlabel('mg/dL'); a1.legend()

gi, gf = DADOS['glucorex']['Glicose_ini'], DADOS['glucorex']['Glicose_fim']
a2.scatter(gi, gf, color='#e76f51'); a2.plot([40, 115], [40, 115], 'k--', lw=1, label='sem efeito')
a2.axhline(40, color='red', ls=':', lw=2, label='piso artificial em 40 mg/dL')
a2.axhline(70, color='orange', ls=':', lw=2, label='limite de hipoglicemia')
a2.set_xlabel('glicose inicial'); a2.set_ylabel('glicose final')
a2.set_title('3 pacientes despencam para exatamente 40,0'); a2.legend()
plt.tight_layout(); plt.show()


**O que isso revela:** o efeito do Glucorex não é uma queda suave de ~5 mg/dL. Ele é:

| Efeito | Nº de pacientes |
|---|---|
| −2,0 mg/dL (efeito típico, leve) | 16 |
| 0,0 (sem efeito) | 8 |
| **+6,0 (glicose sobe)** | 3 |
| **queda brutal até o piso de 40,0 mg/dL** | 3 |

A média de −4,95 é **inteiramente produzida por 3 pacientes que sofreram uma crise hipoglicêmica**. O efeito típico (mediana) é de apenas **−2 mg/dL**. Usar a média aqui é um erro: ela descreve um paciente que não existe.

## 7. Análise de segurança: eventos adversos novos

Média de variável não é a mesma coisa que segurança do paciente. O que importa clinicamente é: **quantos pacientes entraram numa faixa perigosa por causa do remédio** (descontando os que já estavam lá antes).

In [ ]:
LIMITES_CLINICOS = {
    'hipoglicemia (<70)':   ('Glicose',     '<', 70),
    'hiperglicemia (>100)': ('Glicose',     '>', 100),
    'hipotensao (<90)':     ('Pressao',     '<', 90),
    'febre (>37.8)':        ('Temperatura', '>', 37.8),
    'bradicardia (<50)':    ('Frequencia',  '<', 50),
    'taquicardia (>100)':   ('Frequencia',  '>', 100),
}

linhas = []
for med, df in DADOS.items():
    for nome, (var, op, limiar) in LIMITES_CLINICOS.items():
        antes   = df[f'{var}_ini'] < limiar if op == '<' else df[f'{var}_ini'] > limiar
        depois  = df[f'{var}_fim'] < limiar if op == '<' else df[f'{var}_fim'] > limiar
        linhas.append({'Medicamento': med, 'Evento': nome,
                       'No inicio': antes.sum(), 'No fim': depois.sum(),
                       'NOVOS (causados)': (~antes & depois).sum()})

EVENTOS = pd.DataFrame(linhas)
novos = EVENTOS[EVENTOS['NOVOS (causados)'] > 0].sort_values('NOVOS (causados)', ascending=False)
print(f'Eventos adversos NOVOS, causados pelo tratamento (n = 30 por medicamento):\n')
novos


In [ ]:
ax = novos.pivot_table(index='Medicamento', columns='Evento',
                       values='NOVOS (causados)', aggfunc='sum', fill_value=0) \
          .plot(kind='barh', stacked=True, figsize=(11, 4.5), colormap='Set2')
ax.set_xlabel('nº de pacientes que entraram em faixa perigosa (de 30)')
ax.set_title('Eventos adversos causados por cada medicamento')
plt.tight_layout(); plt.show()

print('Gravidade da hipoglicemia induzida pelo Glucorex:')
gl = DADOS['glucorex']
display(gl.loc[gl['Glicose_fim'] < 70, ['Paciente', 'Glicose_ini', 'Glicose_fim', 'Glicose_delta']])


**O Glucorex é o medicamento mais perigoso do conjunto:** ele causa hipoglicemia nova em **5 de 30 pacientes (17%)**, sendo **3 casos graves** (40 mg/dL — nível que exige intervenção médica de emergência).

Já a hiperglicemia causada pela Cardioxina em 6 pacientes é **leve**: a glicose final máxima é 116 mg/dL, ninguém chega perto da faixa diabética (>126).

## 8. Busca de combinações — agora com estatística robusta

Repetimos a busca nas 31 combinações, mas usando a **mediana** (efeito típico) em vez da média, já que vimos que a média do Glucorex é um artefato de 3 outliers. Também comparamos os dois critérios lado a lado para mostrar o impacto da escolha.

In [ ]:
TOLERANCIA = {'Temperatura': 0.20, 'Glicose': 3.00, 'Frequencia': 2.00, 'Energia': 0.50}
QUEDA_MINIMA_PRESSAO = -5.0

def buscar(metrica):
    base = EFEITOS.pivot(index='Medicamento', columns='Variavel', values=metrica)
    out = []
    for n in range(1, len(MEDICAMENTOS) + 1):
        for combo in itertools.combinations(MEDICAMENTOS, n):
            s = base.loc[list(combo)].sum()
            out.append({'Combinacao': ' + '.join(combo),
                        **{v: s[v] for v in VARIAVEIS},
                        'Aprovada': (s['Pressao'] <= QUEDA_MINIMA_PRESSAO)
                                    and all(abs(s[v]) <= t for v, t in TOLERANCIA.items())})
    return pd.DataFrame(out)

por_media   = buscar('delta_medio')
por_mediana = buscar('delta_mediano')

print(f'Aprovadas usando a MEDIA    : {por_media["Aprovada"].sum()}')
print(f'Aprovadas usando a MEDIANA  : {por_mediana["Aprovada"].sum()}   <-- resultado robusto\n')
display(por_media[por_media['Aprovada']])
display(por_mediana[por_mediana['Aprovada']])


In [ ]:
# Por que nenhuma combinacao resolve: quem consegue baixar glicose, e quanto?
efeito_glicose = EFEITOS[EFEITOS['Variavel'] == 'Glicose'] \
                    .set_index('Medicamento')[['delta_medio', 'delta_mediano']]
print('Efeito sobre a glicose (mg/dL):'); display(efeito_glicose.round(2))

melhor = min(
    (sum(efeito_glicose.loc[f, 'delta_mediano'] for f in c), c)
    for n in range(1, 6) for c in itertools.combinations(MEDICAMENTOS, n) if 'cardioxina' in c
)
print(f'A Cardioxina sobe a glicose em +{efeito_glicose.loc["cardioxina","delta_mediano"]:.2f} mg/dL (mediana).')
print(f'Menor residuo alcancavel com qualquer combinacao: {melhor[0]:+.2f} mg/dL, usando {melhor[1]}')
print(f'Tolerancia exigida: +-{TOLERANCIA["Glicose"]:.2f} mg/dL  ->  NENHUMA combinacao anula o efeito colateral.')


In [ ]:
# Comparacao final: Cardioxina sozinha x Cardioxina + Glucorex
comp = pd.DataFrame({
    'Cardioxina sozinha': EFEITOS[EFEITOS.Medicamento == 'cardioxina']
                            .set_index('Variavel')['delta_mediano'],
    'Cardioxina + Glucorex': EFEITOS[EFEITOS.Medicamento.isin(['cardioxina', 'glucorex'])]
                            .groupby('Variavel')['delta_mediano'].sum(),
}).loc[list(VARIAVEIS)]

ax = comp.plot(kind='bar', figsize=(11, 5), color=['#2a9d8f', '#e76f51'])
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('Efeito tipico (mediana)')
ax.set_title('Adicionar Glucorex quase nao mexe na glicose (-2) e traz risco de hipoglicemia grave')
plt.tight_layout(); plt.show()
comp.round(2)


## 9. Conclusão

### Resposta: **Cardioxina** — e nenhuma combinação disponível anula seu efeito colateral

**1. Só a Cardioxina reduz a pressão de forma relevante:** −15,0 mmHg (p < 0,001), levando a média de 130,4 → 115,4 mmHg. O Relaxol reduz apenas −2,0 mmHg e ainda causa bradicardia (−4,6 bpm) e queda de energia. Energozin, Glucorex e Thermocor não alteram a pressão.

**2. O único efeito colateral da Cardioxina é uma alta leve de glicose:** +7,6 mg/dL. É estatisticamente sólido, mas **clinicamente modesto** — a glicose final máxima é 116 mg/dL e nenhum paciente entra em faixa diabética.

**3. Nenhuma combinação anula esse efeito.** Apenas Glucorex e Thermocor baixam a glicose, e o efeito típico somado dos dois é de apenas −2,11 mg/dL contra os +7,69 da Cardioxina: sobra um resíduo de **+5,58 mg/dL**, acima da tolerância. Das 31 combinações, **zero** passam no critério robusto.

**4. E "consertar" com Glucorex sai pior:** o Glucorex causa **hipoglicemia nova em 5 de 30 pacientes (17%), 3 deles graves (40 mg/dL)**. Trocar uma hiperglicemia leve e assintomática por risco de crise hipoglicêmica é uma piora clara do perfil de segurança.

### Recomendação
**Cardioxina em monoterapia**, com monitoramento de glicemia. Se o enunciado exigir efeito colateral *rigorosamente* nulo, a resposta correta é que **nenhum dos tratamentos possíveis atende ao critério** — e apontar isso é mais defensável do que forçar uma combinação que os dados não sustentam.

### A armadilha estatística deste exercício
Usar a **média** do efeito do Glucorex (−4,95 mg/dL) faz "Cardioxina + Glucorex" parecer a resposta perfeita: −15,1 mmHg de pressão e só +2,65 de glicose residual. Mas essa média é produzida por 3 pacientes que despencaram para o piso de 40 mg/dL. O efeito **típico** (mediana) é de apenas −2 mg/dL. A discordância entre t-teste (p = 0,063) e Wilcoxon (p = 0,014) é justamente o alarme que denuncia isso.

> **Lição:** média sem olhar a distribuição transforma um evento adverso grave em aparente benefício terapêutico.

### Limitações
- Efeitos assumidos como **aditivos**; interações farmacológicas reais não estão nos dados.
- n = 30 por medicamento e **sem grupo placebo** — não dá para separar o efeito do fármaco de regressão à média.
- 25 testes sem correção de múltiplas comparações; com Bonferroni (α = 0,002) as conclusões principais (Cardioxina/pressão e Cardioxina/glicose) se mantêm, mas o efeito do Thermocor sobre a frequência cardíaca (p = 0,005) deixa de ser significativo.
- Os limiares de tolerância e as faixas clínicas foram fixados por julgamento; a Seção 8 mostra a sensibilidade do resultado a essa escolha.